In [ ]:
import time
import os
import pyarrow as pa
from IPython.display import display
from utils import (
    fetch_real_time_price,
    arrowify_data,
    calculate_moving_average,
    detect_anomalies,
    save_to_parquet
)

In [ ]:
# Load API key from environment variable or fallback config
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY", "your_api_key_here")

In [ ]:
# Container to hold Arrow tables
table_list = []

# Collect a few real-time price samples
print("Fetching real-time Bitcoin prices...")
for _ in range(5):
    data = fetch_real_time_price(COINGECKO_API_KEY)
    print("Fetched:", data)
    arrow_table = arrowify_data(data)
    table_list.append(arrow_table)
    time.sleep(2)  # wait between samples


In [ ]:
# Combine individual records into one Arrow table
combined_table = pa.concat_tables(table_list)
print("\nCombined Table:")
display(combined_table.to_pandas())

In [ ]:
# Calculate moving average
ma_table = calculate_moving_average(combined_table, window_size=3)
print("\nTable with Moving Average:")
display(ma_table.to_pandas())

In [ ]:
# Detect anomalies
anomaly_table = detect_anomalies(ma_table)
print("\nTable with Anomalies Detected:")
display(anomaly_table.to_pandas())

In [ ]:
# Save to Parquet
save_path = "bitcoin_prices.parquet"
save_to_parquet(anomaly_table, save_path)
print(f"\nSaved processed table to {save_path}")